# 03 - Scene and ArUco marker check

Builds the Track A scene (room, obstacles, charging station with an ArUco marker), renders it, and checks that the marker rendered in MuJoCo is detected by OpenCV. The texture mapping on a box face can be mirrored, which would make the marker undetectable, so both orientations are tried.

In [ ]:
!pip install -q mujoco opencv-contrib-python
!test -d robot || git clone -q https://github.com/hongjiezheng-1-hue/robot.git
!cd robot && git pull -q
!test -d robotis_mujoco_menagerie || git clone -q --depth 1 --filter=blob:none --sparse https://github.com/ROBOTIS-GIT/robotis_mujoco_menagerie.git
!cd robotis_mujoco_menagerie && git sparse-checkout set robotis_tb3

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
import sys
sys.path.insert(0, "robot")
import numpy as np
import cv2
import mujoco
import matplotlib.pyplot as plt
from sim.scene import SceneConfig, write_scene
from perception.aruco import detect_markers

print("OpenCV", cv2.__version__, "| aruco available:", hasattr(cv2, "aruco"))
ROBOT_DIR = "robotis_mujoco_menagerie/robotis_tb3"
cfg = SceneConfig()

In [ ]:
def load(flip):
    path = write_scene(cfg, ROBOT_DIR, flip_marker=flip)
    model = mujoco.MjModel.from_xml_path(path)
    data = mujoco.MjData(model)
    mujoco.mj_forward(model, data)
    return model, data

def render(model, data, lookat, distance, azimuth, elevation, size=(480, 640)):
    renderer = mujoco.Renderer(model, height=size[0], width=size[1])
    cam = mujoco.MjvCamera()
    mujoco.mjv_defaultFreeCamera(model, cam)
    cam.lookat[:] = lookat
    cam.distance, cam.azimuth, cam.elevation = distance, azimuth, elevation
    renderer.update_scene(data, camera=cam)
    return renderer.render().copy()

marker_centre = [cfg.station_x, cfg.station_y, cfg.marker_height]
azimuths = list(range(0, 360, 45))
results = {}
for flip in (False, True):
    model, data = load(flip)
    hits = {}
    for az in azimuths:
        img = render(model, data, marker_centre, 0.8, az, 0)
        ids, _ = detect_markers(img, cfg.marker_dictionary)
        hits[az] = bool(ids is not None and cfg.marker_id in ids.flatten())
    results[flip] = hits
    print(f"flip_marker={flip}: detected at azimuths", [a for a, ok in hits.items() if ok])

In [ ]:
best_flip = max(results, key=lambda f: sum(results[f].values()))
print("orientation to use: flip_marker =", best_flip, "| detections:", sum(results[best_flip].values()), "of", len(azimuths))
model, data = load(best_flip)
top = render(model, data, [0, 0, 0], cfg.room_x * 1.4, 90, -90)
good = [az for az, ok in results[best_flip].items() if ok]
close = render(model, data, marker_centre, 0.8, good[0] if good else 0, 0)
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
axs[0].imshow(top); axs[0].set_title("top-down view"); axs[0].axis("off")
axs[1].imshow(close); axs[1].set_title("marker view"); axs[1].axis("off")
plt.show()

## What to look for

- At least one orientation (`flip_marker` True or False) should be detected from several azimuths. If neither is detected, the texture mapping or lighting needs changing.
- The top-down view should show the room, two obstacles, the robot near the origin and the green charging station against the +x wall.